In [113]:
import os
import yfinance as yf
from google.colab import drive
from pathlib import Path
'''
Script for collecting the six macroeconomic features, four market-return
features, and four lagged stock-return features. It uses FRED for interest rates
and yfinance for market indexes, ETFs, VIX, DXY, copper, and individual stocks.

Feature Sets
6. Macroeconomic Features (6)
7. Market Features (4)
8. Lag Features (4)
'''
# Imports and installs
!pip install yfinance pandas_datareader pandas numpy
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf
from pandas_datareader import data as pdr

START_DATE = "2016-07-01"
END_DATE = "2026-07-01"

OUTPUT_DIRECTORY = Path("/content/drive/MyDrive/Colab Notebooks/CS6140-Final-Project/raw_data")
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
macro_features = download_macro_features(
    START_DATE,
    END_DATE
)

# Semiconductor stocks
tickers = [
    "NVDA", "AMD", "INTC", "QCOM", "AVGO", "TXN",
    "MU", "MRVL", "ADI", "MCHP", "ON", "MPWR"
]

# Output folder
# output_dir = 'raw_data'
drive.mount('/content/drive')
output_dir = '/content/drive/MyDrive/Colab Notebooks/CS6140-Final-Project/raw_data'

# Create folder if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

for ticker in tickers:
    print(f"Downloading {ticker}...")

    df = yf.download(
        ticker,
        # period="10d",      # last 10 trading days
        start=START_DATE,   # past 10 years
        end=END_DATE,
        interval="1d",
        auto_adjust=False,
        actions=True,
    )

    df.reset_index(inplace=True)

    csv_path = os.path.join(
        output_dir,
        f"{ticker}_10y.csv"
    )

    df.to_csv(csv_path, index=False)

    print(f"Saved: {csv_path}")

print("\nDone!")

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved: /content/drive/MyDrive/Colab Notebooks/CS6140-Final-Project/raw_data/NVDA_10y.csv
Saved: /content/drive/MyDrive/Colab Notebooks/CS6140-Final-Project/raw_data/AMD_10y.csv



[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Saved: /content/drive/MyDrive/Colab Notebooks/CS6140-Final-Project/raw_data/INTC_10y.csv
Saved: /content/drive/MyDrive/Colab Notebooks/CS6140-Final-Project/raw_data/QCOM_10y.csv



[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


...

Saved: /content/drive/MyDrive/Colab Notebooks/CS6140-Final-Project/raw_data/MRVL_10y.csv
Saved: /content/drive/MyDrive/Colab Notebooks/CS6140-Final-Project/raw_data/ADI_10y.csv



[*********************100%***********************]  1 of 1 completed


Saved: /content/drive/MyDrive/Colab Notebooks/CS6140-Final-Project/raw_data/MCHP_10y.csv
Saved: /content/drive/MyDrive/Colab Notebooks/CS6140-Final-Project/raw_data/ON_10y.csv


[*********************100%***********************]  1 of 1 completed

Saved: /content/drive/MyDrive/Colab Notebooks/CS6140-Final-Project/raw_data/MPWR_10y.csv

Done!


In [114]:
'''
3. Download macroeconomic features

This function collects:
Effective Federal Funds Rate
10-Year Treasury Yield
VIX
U.S. Dollar Index
Copper Price
Yield Curve: 10-Year minus 2-Year Treasury Yield

The 2-year yield is downloaded because it is needed to calculate the yield
curve, but it does not have to remain as a separate final model feature.
'''

def download_macro_features(
    start_date: str,
    end_date: str
) -> pd.DataFrame:
    """
    Download and construct the six macroeconomic features.

    Final features:
        - fed_funds_rate
        - treasury_10y
        - vix
        - dxy
        - copper
        - yield_curve_10y_2y

    Parameters
    ----------
    start_date:
        First date to download, formatted as YYYY-MM-DD.
    end_date:
        Final date boundary, formatted as YYYY-MM-DD.

    Returns
    -------
    pd.DataFrame
        Daily macroeconomic data indexed by date.
    """

    # FRED series:
    # DFF   = Effective Federal Funds Rate
    # DGS10 = 10-Year Treasury Constant Maturity Rate
    # DGS2  = 2-Year Treasury Constant Maturity Rate
    fred_series = {
        "DFF": "fed_funds_rate",
        "DGS10": "treasury_10y",
        "DGS2": "treasury_2y",
    }

    try:
        fred_data = pdr.DataReader(
            list(fred_series.keys()),
            "fred",
            start_date,
            end_date
        )
    except Exception as error:
        raise RuntimeError(
            "FRED data could not be downloaded. "
            "Check your internet connection and date range."
        ) from error

    fred_data = fred_data.rename(columns=fred_series)

    # Calculate the Treasury yield curve.
    fred_data["yield_curve_10y_2y"] = (
        fred_data["treasury_10y"] - fred_data["treasury_2y"]
    )

    # Yahoo Finance tickers:
    # ^VIX      = CBOE Volatility Index
    # DX-Y.NYB  = U.S. Dollar Index
    # HG=F      = Copper futures
    yahoo_macro_tickers = {
        "^VIX": "vix",
        "DX-Y.NYB": "dxy",
        "HG=F": "copper",
    }

    try:
        yahoo_data = yf.download(
            tickers=list(yahoo_macro_tickers.keys()),
            start=start_date,
            end=end_date,
            auto_adjust=False,
            progress=False,
            group_by="column",
            threads=True
        )
    except Exception as error:
        raise RuntimeError(
            "Yahoo Finance macroeconomic data could not be downloaded."
        ) from error

    yahoo_close = extract_close_prices(yahoo_data)

    yahoo_close = yahoo_close.rename(columns=yahoo_macro_tickers)

    # Combine the FRED and Yahoo Finance data.
    macro_data = fred_data.join(yahoo_close, how="outer")

    # Sort before filling values.
    macro_data = macro_data.sort_index()

    # Economic series and market indexes are not always reported on the same
    # calendar. Forward-fill so that each trading day uses the most recently
    # available observation.
    macro_data = macro_data.ffill()

    # Keep only the final six macro features.
    macro_data = macro_data[
        [
            "fed_funds_rate",
            "treasury_10y",
            "vix",
            "dxy",
            "copper",
            "yield_curve_10y_2y",
        ]
    ]

    macro_data.index = pd.to_datetime(macro_data.index)
    macro_data.index.name = "date"

    return macro_data

In [115]:
'''
The helper below safely extracts close prices because newer versions of yfinance
frequently return a multi-level column index when downloading multiple tickers.
'''
def extract_close_prices(downloaded_data: pd.DataFrame) -> pd.DataFrame:
    """
    Extract Close prices from a yfinance download result.

    Handles both normal and MultiIndex column formats.
    """

    if downloaded_data.empty:
        raise ValueError("The yfinance download returned no rows.")

    if isinstance(downloaded_data.columns, pd.MultiIndex):
        first_level = downloaded_data.columns.get_level_values(0)

        if "Close" not in first_level:
            raise KeyError("The downloaded data does not contain Close prices.")

        close_data = downloaded_data["Close"].copy()
    else:
        if "Close" not in downloaded_data.columns:
            raise KeyError("The downloaded data does not contain a Close column.")

        close_data = downloaded_data[["Close"]].copy()

    if isinstance(close_data, pd.Series):
        close_data = close_data.to_frame()

    return close_data

In [116]:
'''
4. Download market features

This function collects:
Model feature	Yahoo ticker
S&P 500 return	^GSPC
Nasdaq Composite return	^IXIC
SOXX ETF return	SOXX
QQQ ETF return	QQQ

We calculate daily percentage returns instead of using raw index levels because
return values are more comparable across indexes and less affected by long-term
price trends.
'''
def download_market_features(
    start_date: str,
    end_date: str
) -> pd.DataFrame:
    """
    Download market benchmark prices and calculate daily returns.

    Final features:
        - sp500_return
        - nasdaq_return
        - soxx_return
        - qqq_return
    """

    market_tickers = {
        "^GSPC": "sp500_return",
        "^IXIC": "nasdaq_return",
        "SOXX": "soxx_return",
        "QQQ": "qqq_return",
    }

    try:
        downloaded_data = yf.download(
            tickers=list(market_tickers.keys()),
            start=start_date,
            end=end_date,
            auto_adjust=False,
            progress=False,
            group_by="column",
            threads=True
        )
    except Exception as error:
        raise RuntimeError(
            "Market benchmark data could not be downloaded."
        ) from error

    close_prices = extract_close_prices(downloaded_data)

    # Calculate daily decimal returns.
    # For example, 0.012 represents a 1.2% return.
    market_returns = close_prices.pct_change(fill_method=None)

    market_returns = market_returns.rename(columns=market_tickers)

    market_returns = market_returns[
        [
            "sp500_return",
            "nasdaq_return",
            "soxx_return",
            "qqq_return",
        ]
    ]

    market_returns.index = pd.to_datetime(market_returns.index)
    market_returns.index.name = "date"

    return market_returns

In [117]:
'''
5. Download one semiconductor stock

The stock's adjusted close price should be used for return calculations because
it accounts for stock splits and dividend adjustments.
'''

def download_stock_data(
    ticker: str,
    start_date: str,
    end_date: str
) -> pd.DataFrame:
    """
    Download daily OHLCV data for one semiconductor stock.
    """

    try:
        stock_data = yf.download(
            ticker,
            start=start_date,
            end=end_date,
            auto_adjust=False,
            progress=False
        )
    except Exception as error:
        raise RuntimeError(
            f"Data for {ticker} could not be downloaded."
        ) from error

    if stock_data.empty:
        raise ValueError(f"No data was returned for ticker {ticker}.")

    # Remove the ticker level when yfinance returns MultiIndex columns.
    if isinstance(stock_data.columns, pd.MultiIndex):
        stock_data.columns = stock_data.columns.get_level_values(0)

    stock_data = stock_data.rename(
        columns={
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Adj Close": "adjusted_close",
            "Volume": "volume",
        }
    )

    # Some yfinance configurations may omit Adj Close.
    if "adjusted_close" not in stock_data.columns:
        stock_data["adjusted_close"] = stock_data["close"]

    required_columns = [
        "open",
        "high",
        "low",
        "close",
        "adjusted_close",
        "volume",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in stock_data.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{ticker} is missing columns: {missing_columns}"
        )

    stock_data = stock_data[required_columns].copy()
    stock_data.index = pd.to_datetime(stock_data.index)
    stock_data.index.name = "date"

    # Include the company identifier because all 12 stocks will eventually
    # be combined into one panel dataset.
    stock_data["ticker"] = ticker

    return stock_data

In [118]:
'''6. Create the four lag features

The safest implementation uses shift(1). Therefore, a row dated day t only uses
stock returns ending on day t−1, not information from the future
'''

def add_lag_features(stock_data: pd.DataFrame) -> pd.DataFrame:
    """
    Add lagged return features without using future information.

    Features:
        - previous_day_return
        - previous_5_day_return
        - previous_20_day_return
        - previous_60_day_return
    """

    data = stock_data.copy()

    adjusted_close = data["adjusted_close"]

    # Daily stock return for analysis and other feature engineering.
    data["daily_return"] = adjusted_close.pct_change(fill_method=None)

    # The return observed on the previous trading day.
    data["previous_day_return"] = data["daily_return"].shift(1)

    # Returns over windows ending on the previous trading day.
    data["previous_5_day_return"] = (
        adjusted_close.pct_change(5, fill_method=None).shift(1)
    )

    data["previous_20_day_return"] = (
        adjusted_close.pct_change(20, fill_method=None).shift(1)
    )

    data["previous_60_day_return"] = (
        adjusted_close.pct_change(60, fill_method=None).shift(1)
    )

    return data

In [119]:
'''
7. Add the next-month target

The target is not an input feature. It is what your regression models will predict.
'''
def add_target(
    stock_data: pd.DataFrame,
    forecast_horizon: int = 21
) -> pd.DataFrame:
    """
    Create the next-month return target.

    A month is approximated as 21 trading days.

    target = adjusted_close[t + 21] / adjusted_close[t] - 1
    """

    data = stock_data.copy()

    data["target_next_month_return"] = (
        data["adjusted_close"].shift(-forecast_horizon)
        / data["adjusted_close"]
        - 1
    )

    return data

In [120]:
'''
8. Merge everything for one stock
'''
def build_stock_dataset(
    ticker: str,
    macro_features: pd.DataFrame,
    market_features: pd.DataFrame,
    start_date: str,
    end_date: str
) -> pd.DataFrame:
    """
    Build a modeling dataset for one semiconductor company.
    """

    stock_data = download_stock_data(
        ticker=ticker,
        start_date=start_date,
        end_date=end_date
    )

    stock_data = add_lag_features(stock_data)
    stock_data = add_target(stock_data, forecast_horizon=21)

    # Stock trading dates form the main index.
    dataset = stock_data.join(macro_features, how="left")
    dataset = dataset.join(market_features, how="left")

    # Macro values may be unavailable on weekends, holidays, or missing
    # publication dates. Only carry previously observed values forward.
    macro_columns = [
        "fed_funds_rate",
        "treasury_10y",
        "vix",
        "dxy",
        "copper",
        "yield_curve_10y_2y",
    ]

    dataset[macro_columns] = dataset[macro_columns].ffill()

    # Do not backward-fill. Backward filling could place future information
    # into earlier observations.
    dataset = dataset.replace([np.inf, -np.inf], np.nan)

    return dataset

In [121]:
'''
9. Run it for one company
'''

market_features = download_market_features(
    START_DATE,
    END_DATE
)

nvda_dataset = build_stock_dataset(
    ticker="NVDA",
    macro_features=macro_features,
    market_features=market_features,
    start_date=START_DATE,
    end_date=END_DATE
)

print(nvda_dataset.shape)
print(nvda_dataset.head())
print(nvda_dataset.tail())

(2512, 23)
               open     high      low    close  adjusted_close     volume  \
date                                                                        
2016-07-01  1.16875  1.18400  1.16250  1.16650        1.143876  218488000   
2016-07-05  1.16000  1.18575  1.15075  1.18375        1.160791  371084000   
2016-07-06  1.17550  1.19825  1.16950  1.19125        1.168145  273104000   
2016-07-07  1.19925  1.22650  1.19375  1.22225        1.198545  395400000   
2016-07-08  1.23525  1.27725  1.23050  1.27125        1.246594  481932000   

           ticker  daily_return  previous_day_return  previous_5_day_return  \
date                                                                          
2016-07-01   NVDA           NaN                  NaN                    NaN   
2016-07-05   NVDA      0.014788                  NaN                    NaN   
2016-07-06   NVDA      0.006336             0.014788                    NaN   
2016-07-07   NVDA      0.026023             0.006336  

In [122]:
nvda_dataset.to_csv(
    "processed_data/NVDA_model_data.csv",
    index=True
)

In [123]:
semiconductor_tickers = [
    "NVDA",
    "AMD",
    "INTC",
    "AVGO",
    "QCOM",
    "TXN",
    "MU",
    "ADI",
    "MCHP",
    "ON",
    "NXPI",
    "MRVL",
]

In [124]:
'''
10. Run it for all 12 stocks

Replace this list with the exact stocks selected by your group.
'''
all_stock_datasets = []

for ticker in semiconductor_tickers:
    try:
        ticker_dataset = build_stock_dataset(
            ticker=ticker,
            macro_features=macro_features,
            market_features=market_features,
            start_date=START_DATE,
            end_date=END_DATE
        )

        all_stock_datasets.append(ticker_dataset)

        print(
            f"{ticker}: collected {len(ticker_dataset):,} rows"
        )

    except Exception as error:
        print(f"{ticker}: failed - {error}")

combined_dataset = pd.concat(
    all_stock_datasets,
    axis=0
)

combined_dataset = (
    combined_dataset
    .reset_index()
    .sort_values(["date", "ticker"])
    .reset_index(drop=True)
)

print(combined_dataset.shape)
print(combined_dataset.head())

NVDA: collected 2,512 rows
AMD: collected 2,512 rows
INTC: collected 2,512 rows
AVGO: collected 2,512 rows
QCOM: collected 2,512 rows
TXN: collected 2,512 rows
MU: collected 2,512 rows
ADI: collected 2,512 rows
MCHP: collected 2,512 rows
ON: collected 2,512 rows
NXPI: collected 2,512 rows
MRVL: collected 2,512 rows
(30144, 24)
        date       open       high        low      close  adjusted_close  \
0 2016-07-01  56.150002  56.919998  55.849998  56.630001       46.918423   
1 2016-07-01   5.090000   5.140000   5.000000   5.070000        5.070000   
2 2016-07-01  15.447000  15.587000  15.362000  15.422000       11.822212   
3 2016-07-01  32.639999  32.889999  32.470001  32.750000       26.336138   
4 2016-07-01  25.295000  25.360001  25.020000  25.209999       20.872559   

     volume ticker  daily_return  previous_day_return  ...  fed_funds_rate  \
0   1910900    ADI           NaN                  NaN  ...            0.41   
1  18255900    AMD           NaN                  NaN  ...

In [125]:
'''
Build one combined dataset:
'''
all_stock_datasets = []

for ticker in semiconductor_tickers:
    try:
        ticker_dataset = build_stock_dataset(
            ticker=ticker,
            macro_features=macro_features,
            market_features=market_features,
            start_date=START_DATE,
            end_date=END_DATE
        )

        all_stock_datasets.append(ticker_dataset)

        print(
            f"{ticker}: collected {len(ticker_dataset):,} rows"
        )

    except Exception as error:
        print(f"{ticker}: failed - {error}")

combined_dataset = pd.concat(
    all_stock_datasets,
    axis=0
)

combined_dataset = (
    combined_dataset
    .reset_index()
    .sort_values(["date", "ticker"])
    .reset_index(drop=True)
)

print(combined_dataset.shape)
print(combined_dataset.head())

NVDA: collected 2,512 rows
AMD: collected 2,512 rows
INTC: collected 2,512 rows
AVGO: collected 2,512 rows
QCOM: collected 2,512 rows
TXN: collected 2,512 rows
MU: collected 2,512 rows
ADI: collected 2,512 rows
MCHP: collected 2,512 rows
ON: collected 2,512 rows
NXPI: collected 2,512 rows
MRVL: collected 2,512 rows
(30144, 24)
        date       open       high        low      close  adjusted_close  \
0 2016-07-01  56.150002  56.919998  55.849998  56.630001       46.918423   
1 2016-07-01   5.090000   5.140000   5.000000   5.070000        5.070000   
2 2016-07-01  15.447000  15.587000  15.362000  15.422000       11.822212   
3 2016-07-01  32.639999  32.889999  32.470001  32.750000       26.336138   
4 2016-07-01  25.295000  25.360001  25.020000  25.209999       20.872559   

     volume ticker  daily_return  previous_day_return  ...  fed_funds_rate  \
0   1910900    ADI           NaN                  NaN  ...            0.41   
1  18255900    AMD           NaN                  NaN  ...

In [126]:
'''
Save the complete dataset:
'''
import os

combined_dataset.to_csv(
    os.path.join(OUTPUT_DIRECTORY, "semiconductor_model_data.csv"),
    index=False
)

In [127]:
'''
11. Select only these requested features
'''
selected_columns = [
    "date",
    "ticker",

    # Raw stock value used for checking calculations
    "adjusted_close",

    # Six macroeconomic features
    "fed_funds_rate",
    "treasury_10y",
    "vix",
    "dxy",
    "copper",
    "yield_curve_10y_2y",

    # Four market features
    "sp500_return",
    "nasdaq_return",
    "soxx_return",
    "qqq_return",

    # Four lag features
    "previous_day_return",
    "previous_5_day_return",
    "previous_20_day_return",
    "previous_60_day_return",

    # Regression target
    "target_next_month_return",
]

model_data = combined_dataset[selected_columns].copy()

In [128]:
# Remove rows that cannot be used for training:
feature_columns = [
    "fed_funds_rate",
    "treasury_10y",
    "vix",
    "dxy",
    "copper",
    "yield_curve_10y_2y",
    "sp500_return",
    "nasdaq_return",
    "soxx_return",
    "qqq_return",
    "previous_day_return",
    "previous_5_day_return",
    "previous_20_day_return",
    "previous_60_day_return",
]

model_data = model_data.dropna(
    subset=feature_columns + ["target_next_month_return"]
).reset_index(drop=True)

print(model_data.shape)
print(model_data.isna().sum())

(29160, 18)
date                        0
ticker                      0
adjusted_close              0
fed_funds_rate              0
treasury_10y                0
vix                         0
dxy                         0
copper                      0
yield_curve_10y_2y          0
sp500_return                0
nasdaq_return               0
soxx_return                 0
qqq_return                  0
previous_day_return         0
previous_5_day_return       0
previous_20_day_return      0
previous_60_day_return      0
target_next_month_return    0
dtype: int64
